# Segment Tree

Сегодня нужно:

1. понять, **что именно хранит Segment Tree**
2. понять роли методов `__init__`, `_build`, `query` `_query`, `update`, `_update`
3. самостоятельно собрать весь класс
4. проверить реализацию тестами
5. применить структуру к своему индивидуальному проекту

# Зачем нам вообще это

Пусть есть массив:

```text
[2, 5, 1, 4, 3, 7, 6, 8]
```

Нужно поддерживать две операции:

```text
SUM L R — получить сумму на [L..R]
SET i x — заменить a[i] на x
```

Обычный проход:

```text
SUM → O(n)
```

Prefix Sum:

```text
SUM → O(1)
SET → O(n)
```

Но теперь операции постоянно чередуются:

```text
SET
SUM
SET
SUM
...
```

Нам нужна структура, в которой **и запрос, и изменение работают быстро**

# Идея структуры

Мы заранее храним агрегаты для разных кусочков массива:

```text
                        [0..7] = 36
                 /                      \
          [0..3] = 12                [4..7] = 24
          /       \                   /       \
     [0..1]=7   [2..3]=5          [4..5]=10  [6..7]=14
       / \        / \              / \        / \
      2   5       1   4            3   7       6   8
```

Каждая вершина отвечает за свой отрезок

Для суммы значение внутренней вершины получается объединением двух детей:

```text
left + right
```

Ожидаемые сложности:

```text
build  → O(n)
query  → O(log n)
update → O(log n)
```

# Как дерево хранится в памяти

Отдельные объекты `Node` нам не нужны. Мы можем хранить дерево в обычном списке:

```python
tree = [0] * (4 * n)
```

Возьмем некоторое соглашение:

```text
v              — номер текущей вершины
2 * v          — левый ребенок
2 * v + 1      — правый ребенок
```

Кроме номера вершины рекурсивным методам нужно знать, **за какой отрезок она отвечает** (надеюсь это дял вас очевидно):

```text
tl — левая граница сегмента
tr — правая граница сегмента
```

# Что должен делать `__init__`

При создании:

```python
st = SegmentTree(a)
```

объект должен подготовить все необходимое для дальнейшей работы

Продумайте:

- где сохранить длину массива
- нужен ли вам сам исходный массив
- какого размера создать `tree`
- что делать с пустым массивом
- с какой вершины и какого диапазона начать построение

`__init__` сам не должен рекурсивно обходить дерево — для этого есть `_build`

# Что должен делать `_build`

Метод получает:

```text
v, tl, tr
```

и должен сделать так, чтобы:

```text
tree[v]
```

содержал сумму массива на `[tl..tr]`

Подумайте:

- Что хранить, если `[tl..tr]` состоит всего из одного элемента?
- Как разделить более длинный отрезок на две части?
- Какие номера будут у двух детей?
- Когда значения детей уже готовы, как получить значение родителя?
- Где должна закончиться рекурсия?

Ваша цель:

> После завершения `_build(1, 0, n - 1)` корень должен хранить сумму всего массива, а остальные вершины — суммы своих сегментов

# Что должен делать `query`

Внешний метод вызывается просто:

```python
st.query(l, r)
```

Его задача небольшая:

- проверить корректность границ
- обработать пустой массив, если вы хотите его поддерживать
- запустить внутренний `_query` **с корня дерева**

Сам поиск ответа происходит в `_query`

# Что должен делать `_query`

Текущая вершина отвечает за `[tl..tr]`, а нас интересует `[l..r]`

Для каждой вершины нужно определить один из трех случаев:

### 1. Текущий сегмент полностью снаружи запроса

Он ничего не добавляет к сумме

Подумайте, какое значение можно вернуть, чтобы оно **не изменило результат сложения**

### 2. Текущий сегмент полностью внутри запроса

В `tree[v]` уже лежит готовый ответ

Глубже идти не нужно

### 3. Сегменты пересекаются частично

Взять `tree[v]` целиком нельзя

Нужно обратиться к дочерним сегментам и объединить их результаты

# Проверка идеи QUERY руками

Для массива:

```text
[2, 5, 1, 4, 3, 7, 6, 8]
```

найдите:

```text
SUM [2..6]
```

не складывая пять элементов напрямую

Разложите запрос на готовые сегменты дерева:

```text
[...] + [...] + [...] = ...
```

Ожидаемый итог:

```text
21
```

Но важнее самого ответа — **какие вершины дерева вы взяли целиком**

# Что должен делать `update`

Внешний вызов:

```python
st.update(pos, value)
```

должен:

- проверить позицию
- при необходимости обновить сохранённый исходный массив
- запустить внутренний `_update` от корня

Само движение по дереву выполняет `_update`

# Что должен делать `_update`

Изменяется только один элемент массива.

Подумайте:

- В каком листе дерева находится позиция `pos`?
- Как на каждом уровне понять, идти в левого или правого ребенка?
- Что изменить, когда дошли до нужного листа?
- Какие вершины после этого стали неактуальными?
- Как пересчитать вершину после возвращения из рекурсии?
- Нужно ли вообще заходить в соседнюю ветку дерева?

Главная идея:

> Меняем один лист и пересчитываем только его предков до корня

# Теперь соберите весь класс

Не пытайтесь сразу написать все идеально. Удобный порядок:

```text
1. __init__ + _build
2. query + _query
3. update + _update
4. тесты
```

In [ ]:
class SegmentTree:
    def __init__(self, a):
        self.n = len(a)
        self.a = list(a)
        self.tree = [0] * (4 * self.n)
        if self.n > 0:
            self._build(1, 0, self.n - 1)

    def _build(self, v, tl, tr):
        if tl == tr:
            self.tree[v] = self.a[tl]
            return
        tm = (tl + tr) // 2
        self._build(2 * v, tl, tm)
        self._build(2 * v + 1, tm + 1, tr)
        self.tree[v] = self.tree[2 * v] + self.tree[2 * v + 1]

    def query(self, l, r):
        if self.n == 0:
            return 0
        if not (0 <= l <= r < self.n):
            raise IndexError(f"Некорректный диапазон [{l}, {r}] для длины {self.n}")
        return self._query(1, 0, self.n - 1, l, r)

    def _query(self, v, tl, tr, l, r):
        if r < tl or tr < l:
            return 0
        if l <= tl and tr <= r:
            return self.tree[v]
        tm = (tl + tr) // 2
        return (self._query(2 * v, tl, tm, l, r)
                + self._query(2 * v + 1, tm + 1, tr, l, r))

    def update(self, pos, value):
        if not (0 <= pos < self.n):
            raise IndexError(f"Позиция {pos} вне массива длины {self.n}")
        self.a[pos] = value
        self._update(1, 0, self.n - 1, pos, value)

    def _update(self, v, tl, tr, pos, value):
        if tl == tr:
            self.tree[v] = value
            return
        tm = (tl + tr) // 2
        if pos <= tm:
            self._update(2 * v, tl, tm, pos, value)
        else:
            self._update(2 * v + 1, tm + 1, tr, pos, value)
        self.tree[v] = self.tree[2 * v] + self.tree[2 * v + 1]

# Тест 1. Построение

Если `build` работает правильно, корень для этого массива должен хранить сумму всего массива

In [ ]:
a = [2, 5, 1, 4, 3, 7, 6, 8]
st = SegmentTree(a)

assert st.tree[1] == 36
print("BUILD")

BUILD


# Тест 2. Запросы

In [ ]:
a = [2, 5, 1, 4, 3, 7, 6, 8]
st = SegmentTree(a)

tests = [
    (0, 0, 2),
    (0, 3, 12),
    (2, 6, 21),
    (4, 7, 24),
    (0, 7, 36),
    (7, 7, 8),
]

for l, r, expected in tests:
    actual = st.query(l, r)
    assert actual == expected, (
        f"query({l}, {r}) → {actual}, ожидалось {expected}"
    )

print("QUERY")

QUERY


# Тест 3. Обновление

После:

```text
a[3]: 4 → 10
```

сумма всего массива должна увеличиться на `6`

Проверьте также диапазон, содержащий изменённый элемент

In [ ]:
a = [2, 5, 1, 4, 3, 7, 6, 8]
st = SegmentTree(a)

st.update(3, 10)

assert st.query(0, 7) == 42
assert st.query(2, 6) == 27
assert st.query(0, 1) == 7   # этот участок, если че, вообще не менялся

print("UPDATE")

UPDATE


# Тест 4. Несколько изменений подряд

Segment Tree нужен именно для сценария:

```text
update → query → update → query
```

In [ ]:
a = [2, 5, 1, 4, 3, 7, 6, 8]
st = SegmentTree(a)

assert st.query(2, 6) == 21

st.update(3, 10)
assert st.query(2, 6) == 27

st.update(5, 1)
assert st.query(2, 6) == 21

st.update(0, 100)
assert st.query(0, 1) == 105

print("УРА МЫ КРУТЫЕ")

УРА МЫ КРУТЫЕ


# Интеграция в индивидуальный проект

Теперь примените Segment Tree к своему датасету

## Шаг 1. Выберите последовательность

Нужно выбрать:

- поле, задающее порядок записей
- числовое поле, которое потенциально может изменяться

Например:

```text
дата/время → порядок
количество посещений → значение
```

или:

```text
номер события → порядок
стоимость / нагрузка / рейтинг → значение
```

## Шаг 2. Сформулируйте динамический сценарий

Опишите две операции вашей системы:

```text
UPDATE i x
```

— изменить значение одной записи

```text
QUERY L R
```

— получить `SUM`, `MIN` или `MAX` на диапазоне

Сценарий должен иметь смысл именно для ваших данных

## Шаг 3. Реализуйте индекс

Создайте отдельный класс или модуль, например:

```python
class SegmentTreeIndex:
    ...
```

В проекте должно быть видно:

```text
исходные данные
↓
построение Segment Tree
↓
update
↓
range query
```

## Шаг 4. Сравните с наивным решением

Создайте смешанную нагрузку:

```text
update
query
update
query
...
```

Например, не менее `1000` операций

Сравните:

```text
обычный проход по диапазону
vs
Segment Tree
```

Зафиксируйте:

```text
время построения
время обработки всех операций
количество update
количество query
```